<a href="https://colab.research.google.com/github/kamalrawat77/agentic-iam-lab/blob/main/week07-agentic-rag/Nugget036_Multi_Step_Investigation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nugget 036: Multi-Step Investigation Agent

In [1]:
!pip install -q google-genai
import json

In [2]:
from google import genai

def callGPT(prompt):
  client = genai.Client(api_key="APIKEY")
  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )

  return response

In [3]:
def cleanse_response(response):
  clean_response=clean_response = response.text
  clean_response = clean_response.replace("```json", "")
  clean_response = clean_response.replace("```", "")
  clean_response = clean_response.strip()

  return clean_response

In [4]:
def return_json(clean_response,objname):
  responseObj = json.loads(clean_response)
  if not objname:
    return responseObj
  resJsonObj  = responseObj[objname]
  return resJsonObj



Create dummy tools

In [14]:
def trend_analysis():

    return """
Dormant Accounts:
20 → 25 → 27
Trend: Increasing
"""


def search_history():

    return """
Past Investigation:

Root Cause:
Terminated users not processed
"""

In [15]:
question = """
Why are dormant accounts increasing?
"""

In [16]:
trend_result = trend_analysis()

print(trend_result)

history_result = search_history()

print(history_result)


Dormant Accounts:
20 → 25 → 27
Trend: Increasing


Past Investigation:

Root Cause:
Terminated users not processed



In [17]:
evidence = f"""
Trend Analysis:

{trend_result}

Historical Investigation:

{history_result}
"""

In [18]:
prompt = f"""
You are an IAM investigator.

Question:

{question}

Evidence:

{evidence}

Provide:

1. Likely Root Cause
2. Confidence
3. Recommended Action
"""

In [21]:
print(callGPT(prompt).text)

As an IAM Investigator, based on the evidence provided:

---

### **1. Likely Root Cause**

The most likely root cause for the increasing dormant accounts is a **failure in the offboarding process, specifically the incomplete or delayed deprovisioning of accounts belonging to terminated users.**

The historical investigation already identified "Terminated users not processed" as a root cause. Given that the trend of dormant accounts is *still* increasing, it strongly suggests that the underlying issues causing this failure have either recurred, were never fully resolved, or the initial fix was not sustained or comprehensive enough.

### **2. Confidence**

**High.**

The direct correlation between the historical root cause ("Terminated users not processed") and the current problem (increasing dormant accounts) provides a very strong indicator. The evidence points to a consistent, unresolved issue with user lifecycle management, specifically at the termination phase.

### **3. Recommende

Call multiple tools

In [22]:
def dormant_accounts():

    return "Dormant Accounts: 27"


def department_breakdown():

    return """
IT: 45
HR: 12
Finance: 18
"""


def search_history():

    return """
Past Investigation:
Inactive Approvers
"""


def trend_analysis():

    return """
Dormant Accounts:
20 → 25 → 27
Increasing Trend
"""

In [23]:
tools = {
    "dormant_accounts": dormant_accounts,
    "department_breakdown": department_breakdown,
    "search_history": search_history,
    "trend_analysis": trend_analysis
}

In [41]:
executorPrompt = f"""
You are an IAM agent
Question:

{question}

Available Tools:

- trend_analysis
- search_history
- department_breakdown

Return JSON only:

[
  "tool1",
  "tool2"
]

"""

In [45]:
response=callGPT(executorPrompt)

In [46]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""```json
[
  "trend_analysis",
  "department_breakdown"
]
```"""
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-2.5-flash' prompt_feedback=None response_id='vRw8asvVNayDz7IP7b6VyAc' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=23,
  prompt_token_count=59,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=59
    ),
  ],
  thoughts_token_count=517,
  total_token_count=599
) automatic_function_calling_history=[] parsed=None


In [47]:
toolDecision=return_json(cleanse_response(response),None)

Run multiple tools returened by ExecutorPrompt

In [51]:
for tool in toolDecision:

  print("----------------")
  print(tool)
  print("----------------")
  print(tools[tool]())


----------------
trend_analysis
----------------

Dormant Accounts:
20 → 25 → 27
Increasing Trend

----------------
department_breakdown
----------------

IT: 45
HR: 12
Finance: 18

